### "STEM Education: Student Performance in STEM Subjects"


### IMPORT LIBRARIES

In [8]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set visualization style for static plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)


### LOAD DATASET

In [50]:

print("="*80)
print(" STEP 1: LOADING DATASET")
print("="*80)

# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/student_stem_uncleaned_5000.csv')

print(f"✅ Dataset loaded successfully!")
print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n📋 Column names:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i}. {col}")

print(f"\n📄 First 5 rows of the ORIGINAL dataset:")
print(df.head())

print(f"\n📄 Last 5 rows of the ORIGINAL dataset:")
print(df.tail())

print(f"\n📊 Dataset Information:")
print(df.info())

print(f"\n📊 Missing Values Summary:")
print(df.isnull().sum())

 STEP 1: LOADING DATASET
✅ Dataset loaded successfully!
📊 Shape: 5,000 rows × 10 columns

📋 Column names:
   1. student_id
   2. gender
   3. age
   4. study_hours_per_week
   5. attendance_percentage
   6. math_score
   7. science_score
   8. english_score
   9. previous_grade
   10. school_type

📄 First 5 rows of the ORIGINAL dataset:
   student_id gender   age  study_hours_per_week  attendance_percentage  \
0           1  Other  17.0                  21.0                   59.0   
1           2    NaN  18.0                  19.0                   96.0   
2           3   Male  21.0                  32.0                   88.0   
3           4  Other  17.0                   7.0                   83.0   
4           5  Other   NaN                  24.0                   80.0   

   math_score  science_score  english_score previous_grade school_type  
0        98.0           30.0           19.0              C         NaN  
1       100.0           21.0           16.0              B      

### DATA CLEANING

In [10]:

print("\n" + "="*80)
print(" STEP 2: DATA CLEANING")
print("="*80)

# Create a copy of the original dataframe for cleaning
df_clean = df.copy()
# 2.1 HANDLE MISSING VALUES
# ============================================
print("\n" + "-"*60)
print("2.1 HANDLING MISSING VALUES")
print("-"*60)

print("\n🔍 Missing values BEFORE cleaning:")
missing_before = df_clean.isnull().sum()
missing_before = missing_before[missing_before > 0]
print(missing_before)

# Handle categorical columns
print("\n📌 Processing Categorical Columns:")
categorical_cols = ['gender', 'school_type']
for col in categorical_cols:
    if col in df_clean.columns:
        missing_count = df_clean[col].isnull().sum()
        if missing_count > 0:
            print(f"   - {col}: Filling {missing_count} missing values with 'Unknown'")
            df_clean[col] = df_clean[col].fillna('Unknown')

# Handle numeric columns
print("\n📌 Processing Numeric Columns:")
numeric_cols = ['age', 'study_hours_per_week', 'attendance_percentage',
                'math_score', 'science_score', 'english_score']
for col in numeric_cols:
    if col in df_clean.columns:
        missing_count = df_clean[col].isnull().sum()
        if missing_count > 0:
            median_val = df_clean[col].median()
            print(f"   - {col}: Filling {missing_count} missing values with median ({median_val:.2f})")
            df_clean[col] = df_clean[col].fillna(median_val)

# Handle previous_grade
print("\n📌 Processing Previous Grade:")
if 'previous_grade' in df_clean.columns:
    missing_count = df_clean['previous_grade'].isnull().sum()
    if missing_count > 0:
        mode_val = df_clean['previous_grade'].mode()[0]
        print(f"   - previous_grade: Filling {missing_count} missing values with mode ('{mode_val}')")
        df_clean['previous_grade'] = df_clean['previous_grade'].fillna(mode_val)

print("\n✅ Missing values AFTER cleaning:")
missing_after = df_clean.isnull().sum()
missing_after = missing_after[missing_after > 0]
if len(missing_after) == 0:
    print("   🎉 No missing values remaining!")
else:
    print(missing_after)



 STEP 2: DATA CLEANING

------------------------------------------------------------
2.1 HANDLING MISSING VALUES
------------------------------------------------------------

🔍 Missing values BEFORE cleaning:
gender                   1256
age                       548
study_hours_per_week      121
attendance_percentage     105
math_score                 46
science_score              52
english_score              52
previous_grade            811
school_type              1643
dtype: int64

📌 Processing Categorical Columns:
   - gender: Filling 1256 missing values with 'Unknown'
   - school_type: Filling 1643 missing values with 'Unknown'

📌 Processing Numeric Columns:
   - age: Filling 548 missing values with median (19.00)
   - study_hours_per_week: Filling 121 missing values with median (19.00)
   - attendance_percentage: Filling 105 missing values with median (75.00)
   - math_score: Filling 46 missing values with median (51.00)
   - science_score: Filling 52 missing values with medi

In [11]:
# 2.2 FIX INVALID VALUES

print("\n" + "-"*60)
print("2.2 FIXING INVALID VALUES")
print("-"*60)

# Fix negative study_hours_per_week
invalid_hours = len(df_clean[df_clean['study_hours_per_week'] < 0])
if invalid_hours > 0:
    print(f"   - study_hours_per_week: Found {invalid_hours} negative values, replacing with 0")
    df_clean.loc[df_clean['study_hours_per_week'] < 0, 'study_hours_per_week'] = 0

# Fix attendance_percentage > 100 or < 0
invalid_att_high = len(df_clean[df_clean['attendance_percentage'] > 100])
invalid_att_low = len(df_clean[df_clean['attendance_percentage'] < 0])
if invalid_att_high > 0:
    print(f"   - attendance_percentage: Found {invalid_att_high} values > 100, capping at 100")
    df_clean.loc[df_clean['attendance_percentage'] > 100, 'attendance_percentage'] = 100
if invalid_att_low > 0:
    print(f"   - attendance_percentage: Found {invalid_att_low} values < 0, replacing with 0")
    df_clean.loc[df_clean['attendance_percentage'] < 0, 'attendance_percentage'] = 0

# Fix scores > 100 or < 0
for col in ['math_score', 'science_score', 'english_score']:
    if col in df_clean.columns:
        invalid_high = len(df_clean[df_clean[col] > 100])
        invalid_low = len(df_clean[df_clean[col] < 0])
        if invalid_high > 0:
            print(f"   - {col}: Found {invalid_high} values > 100, capping at 100")
            df_clean.loc[df_clean[col] > 100, col] = 100
        if invalid_low > 0:
            print(f"   - {col}: Found {invalid_low} values < 0, replacing with 0")
            df_clean.loc[df_clean[col] < 0, col] = 0

print("\n🔍 Verifying values after fixing:")
validation_summary = {}
for col in ['study_hours_per_week', 'attendance_percentage', 'math_score', 'science_score', 'english_score']:
    if col in df_clean.columns:
        min_val = df_clean[col].min()
        max_val = df_clean[col].max()
        validation_summary[col] = f"Min: {min_val:.1f}, Max: {max_val:.1f}"

for col, summary in validation_summary.items():
    print(f"   - {col}: {summary}")

print("\n✅ All invalid values fixed successfully!")


------------------------------------------------------------
2.2 FIXING INVALID VALUES
------------------------------------------------------------
   - study_hours_per_week: Found 116 negative values, replacing with 0
   - attendance_percentage: Found 100 values > 100, capping at 100
   - math_score: Found 49 values > 100, capping at 100
   - science_score: Found 51 values < 0, replacing with 0

🔍 Verifying values after fixing:
   - study_hours_per_week: Min: 0.0, Max: 39.0
   - attendance_percentage: Min: 50.0, Max: 100.0
   - math_score: Min: 0.0, Max: 100.0
   - science_score: Min: 0.0, Max: 100.0
   - english_score: Min: 0.0, Max: 100.0

✅ All invalid values fixed successfully!


In [12]:
# 2.3 CONVERT CATEGORICAL DATA
# ============================================
print("\n" + "-"*60)
print("2.3 CONVERTING CATEGORICAL DATA")
print("-"*60)

# Gender column standardization
print("\n📌 Standardizing Gender Column:")
print(f"   - Unique values BEFORE: {df_clean['gender'].unique().tolist()}")
gender_mapping = {'M': 'Male', 'F': 'Female', 'O': 'Other', 'Other': 'Other'}
df_clean['gender'] = df_clean['gender'].replace(gender_mapping)
df_clean['gender'] = df_clean['gender'].fillna('Unknown')
print(f"   - Unique values AFTER: {df_clean['gender'].unique().tolist()}")
print(f"   - Value counts:")
gender_counts = df_clean['gender'].value_counts()
for gender, count in gender_counts.items():
    print(f"      • {gender}: {count} ({count/len(df_clean)*100:.1f}%)")

# Convert previous_grade to categorical
print("\n📌 Converting Previous Grade:")
if 'previous_grade' in df_clean.columns:
    df_clean['previous_grade'] = df_clean['previous_grade'].astype('category')
    print(f"   - Unique values: {df_clean['previous_grade'].unique().tolist()}")
    print(f"   - Value counts:")
    grade_counts = df_clean['previous_grade'].value_counts().sort_index()
    for grade, count in grade_counts.items():
        print(f"      • {grade}: {count} ({count/len(df_clean)*100:.1f}%)")

# Convert school_type to categorical
print("\n📌 Converting School Type:")
if 'school_type' in df_clean.columns:
    df_clean['school_type'] = df_clean['school_type'].fillna('Unknown')
    df_clean['school_type'] = df_clean['school_type'].astype('category')
    print(f"   - Unique values: {df_clean['school_type'].unique().tolist()}")
    print(f"   - Value counts:")
    school_counts = df_clean['school_type'].value_counts()
    for school, count in school_counts.items():
        print(f"      • {school}: {count} ({count/len(df_clean)*100:.1f}%)")

print("\n✅ Categorical data conversion complete!")



------------------------------------------------------------
2.3 CONVERTING CATEGORICAL DATA
------------------------------------------------------------

📌 Standardizing Gender Column:
   - Unique values BEFORE: ['Other', 'Unknown', 'Male', 'Female']
   - Unique values AFTER: ['Other', 'Unknown', 'Male', 'Female']
   - Value counts:
      • Male: 1295 (25.9%)
      • Unknown: 1256 (25.1%)
      • Female: 1240 (24.8%)
      • Other: 1209 (24.2%)

📌 Converting Previous Grade:
   - Unique values: ['C', 'B', 'D', 'F', 'A']
   - Value counts:
      • A: 786 (15.7%)
      • B: 818 (16.4%)
      • C: 1710 (34.2%)
      • D: 821 (16.4%)
      • F: 865 (17.3%)

📌 Converting School Type:
   - Unique values: ['Unknown', 'Private', 'Public']
   - Value counts:
      • Private: 1705 (34.1%)
      • Public: 1652 (33.0%)
      • Unknown: 1643 (32.9%)

✅ Categorical data conversion complete!


In [13]:
# Save to Drive
df_clean.to_excel('/content/drive/MyDrive/STEM_Education_Cleaned.xlsx', index=False)
print("✅ Saved to Google Drive")


✅ Saved to Google Drive


In [14]:
# 2.4 CLEANED DATASET PREVIEW

print("\n" + "-"*60)
print("2.4 CLEANED DATASET PREVIEW")
print("-"*60)

print("\n📄 First 5 rows of the CLEANED dataset:")
print(df_clean.head())

print("\n📄 Last 5 rows of the CLEANED dataset:")
print(df_clean.tail())

print("\n📊 Cleaned Dataset Information:")
print(df_clean.info())

print("\n📊 Cleaned Dataset Statistical Summary:")
print(df_clean.describe())

print("\n✅ Data cleaning complete! Total rows:", len(df_clean))
print("="*80)



------------------------------------------------------------
2.4 CLEANED DATASET PREVIEW
------------------------------------------------------------

📄 First 5 rows of the CLEANED dataset:
   student_id   gender   age  study_hours_per_week  attendance_percentage  \
0           1    Other  17.0                  21.0                   59.0   
1           2  Unknown  18.0                  19.0                   96.0   
2           3     Male  21.0                  32.0                   88.0   
3           4    Other  17.0                   7.0                   83.0   
4           5    Other  19.0                  24.0                   80.0   

   math_score  science_score  english_score previous_grade school_type  
0        98.0           30.0           19.0              C     Unknown  
1       100.0           21.0           16.0              B     Unknown  
2        24.0           16.0           50.0              B     Unknown  
3        53.0           83.0           90.0           

### DATA VALIDATION CHECK

In [49]:

print("\n" + "="*80)
print("🔍 STEP 3: DATA VALIDATION")
print("="*80)

print("\n📌 Validating Data Quality:")

# Check for any remaining negative values
negative_cols = []
for col in ['age', 'study_hours_per_week', 'attendance_percentage',
            'math_score', 'science_score', 'english_score']:
    if col in df_clean.columns:
        neg_count = len(df_clean[df_clean[col] < 0])
        if neg_count > 0:
            negative_cols.append(f"{col} ({neg_count})")

if negative_cols:
    print(f"   ⚠️  Negative values found: {', '.join(negative_cols)}")
else:
    print("   ✅ No negative values found in numeric columns")

# Check for attendance > 100
att_high = len(df_clean[df_clean['attendance_percentage'] > 100])
if att_high > 0:
    print(f"   ⚠️  Attendance > 100: {att_high} rows")
else:
    print("   ✅ All attendance values <= 100")

# Check for scores > 100
score_high = 0
for col in ['math_score', 'science_score', 'english_score']:
    if col in df_clean.columns:
        score_high += len(df_clean[df_clean[col] > 100])
if score_high > 0:
    print(f"   ⚠️  Scores > 100: {score_high} rows")
else:
    print("   ✅ All scores <= 100")

# Check for study hours > 168 (max hours in a week)
study_high = len(df_clean[df_clean['study_hours_per_week'] > 168])
if study_high > 0:
    print(f"   ⚠️  Study hours > 168: {study_high} rows")
else:
    print("   ✅ All study hours are reasonable")






🔍 STEP 3: DATA VALIDATION

📌 Validating Data Quality:
   ⚠️  Negative values found: science_score (51)
   ⚠️  Attendance > 100: 100 rows
   ⚠️  Scores > 100: 49 rows
   ✅ All study hours are reasonable


### DATA VISUALISATION (WITH HOVER EFFECT)

### CHART 1: Distribution of All Three Scores

In [48]:


fig1 = make_subplots(rows=1, cols=3, subplot_titles=('Math Score Distribution',
                                                      'Science Score Distribution',
                                                      'English Score Distribution'))

colors = ['#2E86AB', '#A23B72', '#F18F01']
subjects = ['math_score', 'science_score', 'english_score']
titles = ['Math', 'Science', 'English']

for i, (subject, color, title) in enumerate(zip(subjects, colors, titles)):
    hist_data = df_clean[subject].value_counts().sort_index().reset_index()
    hist_data.columns = ['score', 'count']

    fig1.add_trace(
        go.Bar(
            x=hist_data['score'],
            y=hist_data['count'],
            name=title,
            marker_color=color,
            opacity=0.7,
            hovertemplate='<b>Score:</b> %{x}<br><b>Frequency:</b> %{y}<extra></extra>'
        ),
        row=1, col=i+1
    )

    mean_val = df_clean[subject].mean()
    fig1.add_trace(
        go.Scatter(
            x=[mean_val, mean_val],
            y=[0, hist_data['count'].max() * 0.9],
            mode='lines',
            name=f'Mean {title}',
            line=dict(color='red', width=2, dash='dash'),
            hovertemplate=f'<b>Mean {title}:</b> {mean_val:.1f}<extra></extra>'
        ),
        row=1, col=i+1
    )

fig1.update_layout(
    title_text='📊 Distribution of Student Scores Across Subjects',
    height=500,
    showlegend=True,
    template='plotly_white'
)
fig1.show()


### CHART 2: Study Hours vs Math Score

In [46]:

print("\n📊 Creating Chart 2: Study Hours vs Math Score...")

fig2 = px.scatter(
    df_clean,
    x='study_hours_per_week',
    y='math_score',
    color='gender',
    title='📈 Study Hours vs Math Score (Colored by Gender)',
    labels={'study_hours_per_week': 'Study Hours Per Week', 'math_score': 'Math Score'},
    hover_data=['student_id', 'science_score', 'english_score', 'attendance_percentage'],
    opacity=0.6,
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig2.add_trace(
    go.Scatter(
        x=df_clean['study_hours_per_week'].sort_values(),
        y=np.poly1d(np.polyfit(df_clean['study_hours_per_week'], df_clean['math_score'], 1))(
            df_clean['study_hours_per_week'].sort_values()
        ),
        mode='lines',
        name='Trend Line',
        line=dict(color='red', width=2, dash='dash'),
        hovertemplate='<b>Trend:</b> %{y:.1f}<extra></extra>'
    )
)

fig2.update_layout(height=600, template='plotly_white')
fig2.show()



📊 Creating Chart 2: Study Hours vs Math Score...


### CHART 3: Attendance vs Science Score

In [45]:
import pandas as pd
import plotly.express as px

# Ensure study_hours_per_week has no negative values or NaNs before using it for size
df_clean['study_hours_per_week'] = df_clean['study_hours_per_week'].fillna(0).apply(lambda x: max(x, 0))

fig3 = px.scatter(
    df_clean,
    x='attendance_percentage',
    y='science_score',
    color='school_type',
    size='study_hours_per_week',
    title='📈 Attendance vs Science Score (Size = Study Hours)',
    labels={'attendance_percentage': 'Attendance Percentage', 'science_score': 'Science Score'},
    hover_data=['student_id', 'math_score', 'english_score', 'age'],
    opacity=0.6,
    color_discrete_sequence=px.colors.qualitative.Set3
)

fig3.update_layout(height=600, template='plotly_white')
fig3.show()

### CHART 4: Scores by Gender

In [19]:

df_melted = df_clean[df_clean['gender'] != 'Unknown'].melt(
    id_vars=['gender'],
    value_vars=['math_score', 'science_score', 'english_score'],
    var_name='Subject',
    value_name='Score'
)

fig4 = px.box(
    df_melted,
    x='gender',
    y='Score',
    color='Subject',
    title='📊 Score Distribution by Gender',
    labels={'gender': 'Gender', 'Score': 'Score'},
    color_discrete_sequence=px.colors.qualitative.Set1,
    points='all',
    hover_data={'Subject': True, 'Score': True}
)

fig4.update_layout(height=600, template='plotly_white')
fig4.show()


📊 Creating Chart 4: Scores by Gender...


### CHART 5: Correlation Heatmap

In [39]:
corr_matrix = df_clean[['age', 'study_hours_per_week', 'attendance_percentage',
                        'math_score', 'science_score', 'english_score']].corr()

fig7 = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.index,
    colorscale='RdBu',
    zmin=-1,
    zmax=1,
    text=corr_matrix.round(3).values,
    texttemplate='%{text}',
    textfont={"size": 12},
    hoverongaps=False,
    hovertemplate='<b>%{x}</b> vs <b>%{y}</b><br>Correlation: %{z:.3f}<extra></extra>'
))

fig7.update_layout(
    title='📊 Correlation Matrix of Numeric Features',
    height=600,
    template='plotly_white'
)
fig7.show()

### CHART 6: 3D Scatter Plot

In [38]:
import pandas as pd
import plotly.express as px

# Ensure science_score column has no negative values or NaNs, as Plotly's size parameter requires non-negative numeric values.
# First fill NaNs with 0, then ensure all values are non-negative.
sampled_df = df_clean.sample(n=500, random_state=42).copy()
sampled_df['science_score'] = sampled_df['science_score'].fillna(0).apply(lambda x: max(x, 0))

fig8 = px.scatter_3d(
    sampled_df, # Use the sampled_df with guaranteed non-negative and non-NaN science_score
    x='study_hours_per_week',
    y='attendance_percentage',
    z='math_score',
    color='gender',
    size='science_score',
    title='🎯 3D View: Study Hours, Attendance, and Math Score',
    labels={
        'study_hours_per_week': 'Study Hours',
        'attendance_percentage': 'Attendance %',
        'math_score': 'Math Score'
    },
    hover_data=['student_id', 'science_score', 'english_score'],
    color_discrete_sequence=px.colors.qualitative.Set1,
    opacity=0.7
)

fig8.update_layout(height=700, template='plotly_white')
fig8.show()

### CHART 7: Pair Plot Matrix

In [35]:
pair_vars = ['math_score', 'science_score', 'english_score',
             'study_hours_per_week', 'attendance_percentage']

fig9 = px.scatter_matrix(
    df_clean.sample(n=300, random_state=42),
    dimensions=pair_vars,
    color='gender',
    title='📊 Pair Plot Matrix of Key Variables',
    opacity=0.6,
    color_discrete_sequence=px.colors.qualitative.Set2,
    hover_data=['student_id']
)

fig9.update_layout(height=800, template='plotly_white')
fig9.show()

### CHART 8: Violin Plot - Math Score by Gender

In [34]:
fig10 = px.violin(
    df_clean[df_clean['gender'] != 'Unknown'],
    x='gender',
    y='math_score',
    color='gender',
    title='🎻 Math Score Distribution by Gender (Violin Plot)',
    labels={'gender': 'Gender', 'math_score': 'Math Score'},
    box=True,
    points='all',
    hover_data={'gender': True, 'math_score': True},
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig10.update_layout(height=600, template='plotly_white')
fig10.show()


### CHART 9: Gender Distribution Pie Chart

In [33]:
gender_counts = df_clean['gender'].value_counts().reset_index()
gender_counts.columns = ['gender', 'count']

fig12 = px.pie(
    gender_counts,
    values='count',
    names='gender',
    title='👤 Gender Distribution of Students',
    hover_data={'gender': True, 'count': True},
    color_discrete_sequence=px.colors.qualitative.Set2,
    hole=0.3
)

fig12.update_traces(textposition='inside', textinfo='percent+label')
fig12.update_layout(height=500, template='plotly_white')
fig12.show()


### **CHART 10:School Type Distribution Pie Chart**

In [32]:
school_counts = df_clean['school_type'].value_counts().reset_index()
school_counts.columns = ['school_type', 'count']

fig13 = px.pie(
    school_counts,
    values='count',
    names='school_type',
    title='🏫 School Type Distribution',
    hover_data={'school_type': True, 'count': True},
    color_discrete_sequence=px.colors.qualitative.Set3,
    hole=0.3
)

fig13.update_traces(textposition='inside', textinfo='percent+label')
fig13.update_layout(height=500, template='plotly_white')
fig13.show()


### CHART 11: Previous Grade by School Type

In [31]:
grade_school = df_clean[df_clean['school_type'] != 'Unknown'].groupby(
    ['school_type', 'previous_grade']).size().reset_index(name='count')

fig14 = px.bar(
    grade_school,
    x='school_type',
    y='count',
    color='previous_grade',
    title='📊 Previous Grade Distribution by School Type',
    labels={'school_type': 'School Type', 'count': 'Number of Students', 'previous_grade': 'Previous Grade'},
    barmode='stack',
    color_discrete_sequence=px.colors.qualitative.Set2,
    hover_data={'school_type': True, 'previous_grade': True, 'count': True}
)

fig14.update_layout(height=600, template='plotly_white')
fig14.show()


### **Comprehensiv**e **Dashboard**

In [30]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Ensure df_clean is defined, in case previous cells were not run
# Note: This will use the uncleaned version of df if cleaning cells haven't been executed.
if 'df' in locals():
    df_clean = df.copy()
else:
    # If df is also not defined, load it from the source
    print("Warning: 'df' not found, attempting to load original dataset.")
    try:
        df = pd.read_csv('/content/drive/MyDrive/student_stem_uncleaned_5000.csv')
        df_clean = df.copy()
    except FileNotFoundError:
        print("Error: Original dataset '/content/drive/MyDrive/student_stem_uncleaned_5000.csv' not found. Cannot proceed.")
        df_clean = pd.DataFrame() # Create an empty DataFrame to avoid further errors

fig15 = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Study Hours Distribution',
        'Attendance Distribution',
        'Math Score by Study Hours (Binned)',
        'Average Scores by Age Group'
    )
)

# 1. Study Hours Distribution
study_hours_bins = pd.cut(df_clean['study_hours_per_week'], bins=10)
study_hours_counts = study_hours_bins.value_counts().sort_index().reset_index()
study_hours_counts.columns = ['Range', 'Count']
study_hours_counts['Range'] = study_hours_counts['Range'].astype(str)

fig15.add_trace(
    go.Bar(
        x=study_hours_counts['Range'],
        y=study_hours_counts['Count'],
        marker_color='#2E86AB',
        opacity=0.7,
        name='Study Hours',
        hovertemplate='<b>Hours Range:</b> %{x}<br><b>Students:</b> %{y}<extra></extra>'
    ),
    row=1, col=1
)

# 2. Attendance Distribution
attendance_bins = pd.cut(df_clean['attendance_percentage'], bins=10)
attendance_counts = attendance_bins.value_counts().sort_index().reset_index()
attendance_counts.columns = ['Range', 'Count']
attendance_counts['Range'] = attendance_counts['Range'].astype(str)

fig15.add_trace(
    go.Bar(
        x=attendance_counts['Range'],
        y=attendance_counts['Count'],
        marker_color='#A23B72',
        opacity=0.7,
        name='Attendance',
        hovertemplate='<b>Attendance Range:</b> %{x}<br><b>Students:</b> %{y}<extra></extra>'
    ),
    row=1, col=2
)

# 3. Math Score by Study Hours (Binned)
df_clean['study_hours_bin'] = pd.cut(df_clean['study_hours_per_week'], bins=5)
avg_math_by_hours = df_clean.groupby('study_hours_bin', observed=False)['math_score'].mean().reset_index()
avg_math_by_hours['study_hours_bin'] = avg_math_by_hours['study_hours_bin'].astype(str)

fig15.add_trace(
    go.Bar(
        x=avg_math_by_hours['study_hours_bin'],
        y=avg_math_by_hours['math_score'],
        marker_color='#F18F01',
        opacity=0.7,
        name='Avg Math Score',
        hovertemplate='<b>Hours Range:</b> %{x}<br><b>Avg Math Score:</b> %{y:.1f}<extra></extra>'
    ),
    row=2, col=1
)

# 4. Average Scores by Age Group
df_clean['age_group'] = pd.cut(df_clean['age'], bins=[14, 16, 18, 20, 22, 24],
                                labels=['14-16', '16-18', '18-20', '20-22', '22-24'])
avg_by_age = df_clean.groupby('age_group', observed=False)[['math_score', 'science_score', 'english_score']].mean().reset_index()
avg_by_age_melted = avg_by_age.melt(
    id_vars=['age_group'],
    value_vars=['math_score', 'science_score', 'english_score'],
    var_name='Subject',
    value_name='Score'
)

fig15.add_trace(
    go.Scatter(
        x=avg_by_age_melted[avg_by_age_melted['Subject'] == 'math_score']['age_group'],
        y=avg_by_age_melted[avg_by_age_melted['Subject'] == 'math_score']['Score'],
        mode='lines+markers',
        name='Math',
        line=dict(color='#2E86AB', width=3),
        marker=dict(size=10),
        hovertemplate='<b>Age:</b> %{x}<br><b>Math Score:</b> %{y:.1f}<extra></extra>'
    ),
    row=2, col=2
)

fig15.add_trace(
    go.Scatter(
        x=avg_by_age_melted[avg_by_age_melted['Subject'] == 'science_score']['age_group'],
        y=avg_by_age_melted[avg_by_age_melted['Subject'] == 'science_score']['Score'],
        mode='lines+markers',
        name='Science',
        line=dict(color='#A23B72', width=3),
        marker=dict(size=10),
        hovertemplate='<b>Age:</b> %{x}<br><b>Science Score:</b> %{y:.1f}<extra></extra>'
    ),
    row=2, col=2
)

fig15.add_trace(
    go.Scatter(
        x=avg_by_age_melted[avg_by_age_melted['Subject'] == 'english_score']['age_group'],
        y=avg_by_age_melted[avg_by_age_melted['Subject'] == 'english_score']['Score'],
        mode='lines+markers',
        name='English',
        line=dict(color='#F18F01', width=3),
        marker=dict(size=10),
        hovertemplate='<b>Age:</b> %{x}<br><b>English Score:</b> %{y:.1f}<extra></extra>'
    ),
    row=2, col=2
)

fig15.update_layout(
    title='📊 Comprehensive EDA Dashboard',
    height=800,
    showlegend=True,
    template='plotly_white'
)

fig15.show()

### **FINAL** **INSIGHTS**

In [28]:
print("\n" + "="*80)
print("📊 FINAL SUMMARY AND KEY INSIGHTS")
print("="*80)

print("\n📈 BASIC STATISTICS:")
print(f"   Total Students: {len(df_clean):,}")
print(f"\n   Average Scores:")
print(f"      - Math: {df_clean['math_score'].mean():.2f} (±{df_clean['math_score'].std():.2f})")
print(f"      - Science: {df_clean['science_score'].mean():.2f} (±{df_clean['science_score'].std():.2f})")
print(f"      - English: {df_clean['english_score'].mean():.2f} (±{df_clean['english_score'].std():.2f})")

print(f"\n   Average Study Hours: {df_clean['study_hours_per_week'].mean():.2f} hrs/week")
print(f"   Average Attendance: {df_clean['attendance_percentage'].mean():.2f}%")

print("\n👤 GENDER DISTRIBUTION:")
for gender in df_clean['gender'].unique():
    if gender != 'Unknown':
        count = len(df_clean[df_clean['gender'] == gender])
        print(f"   - {gender}: {count:,} students ({count/len(df_clean)*100:.1f}%)")

print("\n🏫 SCHOOL TYPE DISTRIBUTION:")
for school in df_clean['school_type'].unique():
    if school != 'Unknown':
        count = len(df_clean[df_clean['school_type'] == school])
        print(f"   - {school}: {count:,} students ({count/len(df_clean)*100:.1f}%)")

print("\n📊 CORRELATION INSIGHTS:")
corr_with_math = df_clean[['math_score', 'science_score', 'english_score',
                           'study_hours_per_week', 'attendance_percentage']].corr()['math_score']
for feature, corr in corr_with_math.items():
    if feature != 'math_score':
        strength = "Strong" if abs(corr) > 0.5 else "Moderate" if abs(corr) > 0.3 else "Weak"
        direction = "positive" if corr > 0 else "negative"
        print(f"   - {feature} vs Math: {corr:.3f} ({strength} {direction} correlation)")

print("\n" + "="*80)
print(" ALL 15 INTERACTIVE CHARTS DISPLAYED SUCCESSFULLY!")
print(" Hover over any chart element to see detailed information")
print(" Cleaned dataset saved in variable 'df_clean'")
print("="*80)


📊 FINAL SUMMARY AND KEY INSIGHTS

📈 BASIC STATISTICS:
   Total Students: 5,000

   Average Scores:
      - Math: 51.29 (±30.65)
      - Science: 48.97 (±29.80)
      - English: 50.21 (±29.44)

   Average Study Hours: 18.85 hrs/week
   Average Attendance: 75.74%

👤 GENDER DISTRIBUTION:
   - Other: 1,209 students (24.2%)
   - nan: 0 students (0.0%)
   - Male: 1,295 students (25.9%)
   - Female: 1,240 students (24.8%)

🏫 SCHOOL TYPE DISTRIBUTION:
   - nan: 0 students (0.0%)
   - Private: 1,705 students (34.1%)
   - Public: 1,652 students (33.0%)

📊 CORRELATION INSIGHTS:
   - science_score vs Math: -0.031 (Weak negative correlation)
   - english_score vs Math: -0.000 (Weak negative correlation)
   - study_hours_per_week vs Math: -0.034 (Weak negative correlation)
   - attendance_percentage vs Math: 0.031 (Weak positive correlation)

 ALL 15 INTERACTIVE CHARTS DISPLAYED SUCCESSFULLY!
 Hover over any chart element to see detailed information
 Cleaned dataset saved in variable 'df_clean'
